In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [2]:
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path().resolve().parent
ROOT_SRC = ROOT / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT:", ROOT)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *

ROOT: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [3]:
root0 = ROOT / "data"
gdc = GDC(root0=root0)

os.listdir(root0)[:10]

['cancer',
 'reactome_summary.tsv',
 'reactome',
 'vector_store',
 'reactome_pathways.tsv',
 'TCGA',
 'gdc_programs.txt']

### Methods

#### Get GDC programs - get_gdc_progams()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "facets": "program.name"  

#### Get valid primary sites - get_primary_sites()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "fields": "project_id,name,primary_site,disease_type"  

#### Get valid subtypes - get_subtypes()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> facets = "diagnoses.primary_diagnosis"  

#### For each subtype → get stages  - get_stages()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> "field": "diagnoses.ajcc_pathologic_stage" - AJCC diagnoses   

#### For each (subtype, stage) → get_samples()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> given pid, subtype, stage  
> from cases   
> "samples.sample_id", "samples.submitter_id", "samples.sample_type"  

#### get RNA-seq files - get_expression_files_given_samples()

> given: "field": "cases.case_id", "value": case_ids  
> end_point: url_gdc_files = "https://api.gdc.cancer.gov/files"  



### Get all programs

In [4]:
force = False
verbose = True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

np.array(prog_list)

File read at '/home/flavio/uv/perturb_agent/data/gdc_programs.txt'


array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [5]:
gdc.url_gdc_project

'https://api.self.cancer.gov/projects'

In [6]:
force = False
verbose = True

prog_id = "TCGA"
df_psi = gdc.get_primary_sites(prog_id=prog_id, force=force, verbose=verbose)

psi_id_list = np.unique(df_psi.psi_id)
psi_id_list

Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/TCGA/primary_site_program_TCGA.tsv'


array(['TCGA-ACC', 'TCGA-BLCA', 'TCGA-BRCA', 'TCGA-CESC', 'TCGA-CHOL',
       'TCGA-COAD', 'TCGA-DLBC', 'TCGA-ESCA', 'TCGA-GBM', 'TCGA-HNSC',
       'TCGA-KICH', 'TCGA-KIRC', 'TCGA-KIRP', 'TCGA-LAML', 'TCGA-LGG',
       'TCGA-LIHC', 'TCGA-LUAD', 'TCGA-LUSC', 'TCGA-MESO', 'TCGA-OV',
       'TCGA-PAAD', 'TCGA-PCPG', 'TCGA-PRAD', 'TCGA-READ', 'TCGA-SARC',
       'TCGA-SKCM', 'TCGA-STAD', 'TCGA-TGCT', 'TCGA-THCA', 'TCGA-THYM',
       'TCGA-UCEC', 'TCGA-UCS', 'TCGA-UVM'], dtype=object)

In [7]:
df_psi.tail(3)

,psi_id,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

- Brain tumors do NOT use AJCC staging (3 - TCGA-LGG Brain)

In [8]:
psi_id = "TCGA-LUAD"

gdc.set_primary_site(psi_id=psi_id)

primary_site = gdc.primary_site
psi_id = gdc.psi_id

psi_id, primary_site

('TCGA-LUAD', 'Bronchus and lung')

In [9]:
force = False
verbose = True

print(psi_id, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(
    batch_size=200, do_filter=False, force=force, verbose=verbose
)

print("df_cases", len(df_cases), "df_subt", len(df_subt))

TCGA-LUAD Bronchus and lung
Table opened ((585, 23)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-LUAD/cases_for_TCGA-LUAD.tsv'
df_cases 585 df_subt 23


### df_subt

In [10]:
df_subt

,psi_id,subtype_global,tumor_class,subtype_tissue,stage,n
0,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,unknown,136
1,TCGA-LUAD,other,other,other,unknown,136
2,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA,76
3,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IB,71
4,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIIA,40
5,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIB,37
6,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIA,27
7,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IV,14
8,TCGA-LUAD,other,other,other,Stage IB,9
9,TCGA-LUAD,other,other,other,Stage IA,8


### df_cases

In [11]:
df_cases.head(3)

,primary_site,disease_type,case_id,diagnoses,psi_id,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,primary_site_norm,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac
0,Bronchus and lung,Adenomas and Adenocarcinomas,07b5663f-9a54-4462-b6c1-6fc8116b8714,"[{'primary_diagnosis': 'Adenocarcinoma, NOS', ...",TCGA-LUAD,adenocarcinoma_generic,Stage IA,"Adenocarcinoma, NOS",NaN,NaN,...,bronchus and lung,adenomas and adenocarcinomas,adenocarcinoma,adenocarcinoma,epithelial,lung_adenocarcinoma,ok,valid,1,0.001709
1,Bronchus and lung,Adenomas and Adenocarcinomas,294ff941-aea1-4588-9a0e-e9f5393e2bb6,[{'primary_diagnosis': 'Infiltrating duct carc...,TCGA-LUAD,other,Stage IA,"Infiltrating duct carcinoma, NOS",NaN,NaN,...,bronchus and lung,adenomas and adenocarcinomas,infiltrating duct carcinoma,other,other,other,ok,ambiguous,1,0.001709
2,Bronchus and lung,Adenomas and Adenocarcinomas,ebb33753-9d38-4368-9033-3e55f129d00d,[{'primary_diagnosis': 'Adenocarcinoma with mi...,TCGA-LUAD,adenocarcinoma_generic,unknown,Adenocarcinoma with mixed subtypes,NaN,NaN,...,bronchus and lung,adenomas and adenocarcinomas,adenocarcinoma with mixed subtypes,adenocarcinoma,epithelial,lung_adenocarcinoma,ok,valid,1,0.001709


In [12]:
cols = ["psi_id", "case_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"]

df_cases[cols].head(12)

,psi_id,case_id,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-LUAD,07b5663f-9a54-4462-b6c1-6fc8116b8714,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
1,TCGA-LUAD,294ff941-aea1-4588-9a0e-e9f5393e2bb6,other,other,other,Stage IA
2,TCGA-LUAD,ebb33753-9d38-4368-9033-3e55f129d00d,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,unknown
3,TCGA-LUAD,781f40c9-c099-4c96-8269-ebe2a449c93d,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIB
4,TCGA-LUAD,5134c56f-8286-4ec8-8348-237cee7dad5e,other,other,other,Stage IIA
5,TCGA-LUAD,369f14c4-2191-4962-a309-3e23ddc4e5fc,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIA
6,TCGA-LUAD,e51f717f-72e8-400e-9f35-fc5dc32fdf71,other,other,other,unknown
7,TCGA-LUAD,bbe88801-34f3-46d2-bbfd-b46c3901ed71,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IB
8,TCGA-LUAD,d2c1e896-6886-4122-bb48-5fbcd3f641f4,other,other,other,unknown
9,TCGA-LUAD,108a71cf-b9db-47cd-aa74-c03ec989b41b,other,other,other,Stage IIA


### Get all cases and their classifications

In [13]:
force = False
verbose = False

run_all = False

if run_all:
    for i, row in df_psi.iterrows():
        print(i, end=" ")
        psi_id = row.psi_id

        gdc.set_primary_site(psi_id=psi_id)

        print(psi_id, primary_site)

        df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(
            batch_size=200, do_filter=False, force=force, verbose=verbose
        )

### Has tumors

In [14]:
force = False
verbose = False

dic = {}
for psi_id in gdc.df_psi.psi_id:
    _, df_tumor_samples, _, _ = gdc.get_filtered_tables(
        psi_id=psi_id, sample_type_term="Primary Tumor", verbose=verbose
    )

    dic[psi_id] = True if len(df_tumor_samples) > 2 else False

df_hastumor_samples = pd.DataFrame.from_dict(dic, orient="index", columns=["has_tumor_samples"])
df_hastumor_samples.head(6)

,has_tumor_samples
TCGA-ACC,True
TCGA-PCPG,True
TCGA-BLCA,True
TCGA-LGG,False
TCGA-GBM,True
TCGA-BRCA,True


In [15]:
df_hastumor_samples[df_hastumor_samples.has_tumor_samples == True]

,has_tumor_samples
TCGA-ACC,True
TCGA-PCPG,True
TCGA-BLCA,True
TCGA-GBM,True
TCGA-BRCA,True
TCGA-LUAD,True
TCGA-LUSC,True
TCGA-MESO,True
TCGA-CESC,True
TCGA-COAD,True


### Are there normal samples?

In [16]:
force = False
verbose = False

dic = {}
for psi_id in gdc.df_psi.psi_id:
    _, df_normal_samples, _, _ = gdc.get_filtered_tables(
        psi_id=psi_id, sample_type_term="Solid Tissue Normal", verbose=verbose
    )

    dic[psi_id] = True if len(df_normal_samples) > 2 else False

df_hasnormal_samples = pd.DataFrame.from_dict(dic, orient="index", columns=["has_normal_samples"])
df_hasnormal_samples.head(6)

,has_normal_samples
TCGA-ACC,True
TCGA-PCPG,False
TCGA-BLCA,True
TCGA-LGG,False
TCGA-GBM,False
TCGA-BRCA,True


In [17]:
df_hasnormal_samples[df_hasnormal_samples.has_normal_samples == True]

,has_normal_samples
TCGA-ACC,True
TCGA-BLCA,True
TCGA-BRCA,True
TCGA-LUAD,True
TCGA-LUSC,True
TCGA-CESC,True
TCGA-COAD,True
TCGA-READ,True
TCGA-ESCA,True
TCGA-KICH,True


### Encapsulated - calc_degs()

In [18]:
";".join(gdc.df_psi.psi_id)

'TCGA-ACC;TCGA-PCPG;TCGA-BLCA;TCGA-LGG;TCGA-GBM;TCGA-BRCA;TCGA-LUAD;TCGA-LUSC;TCGA-MESO;TCGA-CESC;TCGA-COAD;TCGA-READ;TCGA-DLBC;TCGA-SKCM;TCGA-ESCA;TCGA-UVM;TCGA-LAML;TCGA-KICH;TCGA-KIRP;TCGA-KIRC;TCGA-HNSC;TCGA-LIHC;TCGA-CHOL;TCGA-PAAD;TCGA-PRAD;TCGA-SARC;TCGA-OV;TCGA-STAD;TCGA-TGCT;TCGA-THYM;TCGA-THCA;TCGA-UCS;TCGA-UCEC'

In [19]:
lfc_cutoff = 1.0
fdr_cutoff = 0.05
force = False
verbose = False

dic = {}
for psi_id in gdc.df_psi.psi_id:
    print(">>>", psi_id)
    df_degs, df_lfc, degs_txt = gdc.calc_degs(
        psid_id=psi_id,
        root_src=ROOT_SRC,
        lfc_cutoff=lfc_cutoff,
        fdr_cutoff=fdr_cutoff,
        verbose=verbose,
        force=force,
    )

    dic[psi_id] = True if len(df_degs) > 0 else False

df = pd.DataFrame.from_dict(dic, orient="index", columns=["has_degs"])
df.head(6)

>>> TCGA-ACC
>>> TCGA-PCPG
>>> TCGA-BLCA
0.........10. -> 12
Dowloading tumor files: 0.........10. -> 12
>>> TCGA-LGG
>>> TCGA-GBM
>>> TCGA-BRCA
>>> TCGA-LUAD
>>> TCGA-LUSC
0.........10.........20.........30.........40.........50.... -> 55
Dowloading tumor files: 0.........10.........20.........30.........40.........50.... -> 55
>>> TCGA-MESO
>>> TCGA-CESC
0 -> 1
Dowloading tumor files: 0 -> 1
>>> TCGA-COAD
0.........10.........20.........30..... -> 36
Dowloading tumor files: 0.........10.........20.........30..... -> 36
>>> TCGA-READ
>>> TCGA-DLBC
>>> TCGA-SKCM
>>> TCGA-ESCA
0.........10.........20.........30 -> 31
Dowloading tumor files: 0.........10.........20.........30 -> 31
>>> TCGA-UVM
>>> TCGA-LAML
>>> TCGA-KICH
>>> TCGA-KIRP
0.........10.........20......... -> 30
Dowloading tumor files: 0.........10.........20......... -> 30
>>> TCGA-KIRC
0.........10.........20.. -> 23
Dowloading tumor files: 0.........10.........20.. -> 23
>>> TCGA-HNSC
0.........10.........20... -> 24
Dowlo

,has_degs
TCGA-ACC,False
TCGA-PCPG,False
TCGA-BLCA,False
TCGA-LGG,False
TCGA-GBM,False
TCGA-BRCA,False


In [20]:
df[df.has_degs == True]

,has_degs
TCGA-LUAD,True


In [23]:
lfc_cutoff = 1.0
fdr_cutoff = 0.05
force = False
verbose = False

psi_id = "TCGA-LUSC"
psi_id = "TCGA-BRCA"
psi_id = "TCGA-LUAD"

df_degs, df_lfc, degs_txt = gdc.calc_degs(
    psid_id=psi_id,
    root_src=ROOT_SRC,
    lfc_cutoff=lfc_cutoff,
    fdr_cutoff=fdr_cutoff,
    verbose=verbose,
    force=force,
)

print(len(df_degs), len(df_lfc))
df_degs.head(6)

3978 19099


,gene_id,symbol,gene_type,lfc,logCPM,statistic,pvalue,fdr,method
0,ENSG00000263585,AC145207.4,lncRNA,4.507180,5.232704,225.007116,4.156266e-14,3.969027e-10,edgeR_QLF
1,ENSG00000253258,AC068228.1,lncRNA,7.780715,1.799834,125.369618,7.105894e-14,4.523849e-10,edgeR_QLF
2,ENSG00000204949,FAM83A-AS1,lncRNA,8.034693,1.445862,121.460858,3.757829e-12,7.226333e-09,edgeR_QLF
3,ENSG00000251018,HMMR-AS1,lncRNA,3.867438,0.155476,114.735365,3.783618e-12,7.226333e-09,edgeR_QLF
4,ENSG00000153294,ADGRF4,protein_coding,5.681105,1.702539,108.743143,6.189391e-12,9.850932e-09,edgeR_QLF
5,ENSG00000128050,PAICS,protein_coding,2.399840,6.492267,138.590405,8.823075e-12,1.296245e-08,edgeR_QLF


### Build tumor and normal dictionary

In [ ]:
force = False
verbose = False

psi_id = "TCGA-LUAD"

dic_tumor, dic_normal = gdc.get_file_expression_tumor_and_normal(
    psi_id=psi_id, force=force, verbose=verbose
)

len(dic_tumor), len(dic_normal)

### Get Expression table (for tumor and control)

#### 🧪 Mental model (important)

| Step	|  Needs strand? | Output | Column |
|---------|---------|---------|---------|
| Read assignment	|  YES |  counts | stranded_first |
| Normalization	| NO | TPM / FPKM | both: unstranded |
| Biological meaning | NO | gene expression| both: unstranded |

### Merge Expression Tables for DEGs' calculation

In [ ]:
verbose = False
dfa_tumor, dfa_normal = gdc.merge_normal_tumor_tables(
    dic_tumor, dic_normal, imax_tumor=12, imax_normal=12, verbose=verbose
)

In [ ]:
dfa_normal.head(3)

In [ ]:
dfa_tumor.head(3)

### Calc DEGs

In [ ]:
gdc.root_psi

In [ ]:
cdegs = CALC_DEGS(root_psi=gdc.root_psi, src_dir=ROOT_SRC)

cdegs.libs_dir, cdegs.root_psi

In [ ]:
dfa_normal2 = cdegs.deduplicate_by_max_reads(dfa_normal)
len(dfa_normal2), len(dfa_normal)

In [ ]:
dfa_tumor2 = cdegs.deduplicate_by_max_reads(dfa_tumor)
len(dfa_tumor2), len(dfa_tumor)

In [ ]:
df_lfc = cdegs.run_deg_rscript(
    df_tumor=dfa_tumor2,
    df_normal=dfa_normal2,
    method="edger",
    manual_dispersion=0.1,
    min_total_count=10,
    merge_how="inner",
    keep_temp=False,
)
print(len(df_lfc))

In [ ]:
df_lfc = df_lfc.rename(columns={"log2FoldChange": "lfc", "padj": "fdr"})
df_lfc.head(3)

In [ ]:
fname = f"lfc_{gdc.psi_id}.tsv"
pdwritecsv(df_lfc, fname, gdc.root_psi)

In [ ]:
lfc_cutoff = 1.0
fdr_cutoff = 0.05

df_degs = df_lfc[(df_lfc.lfc >= lfc_cutoff) & (df_lfc.fdr < fdr_cutoff)].copy()
print(len(df_degs))

df_degs.head(3)

In [ ]:
str(gdc.root_psi)

In [ ]:
fname = f"degs_{gdc.psi_id}.txt"

text = "\n".join(df_degs.symbol)

write_txt(text, fname, str(gdc.root_psi))

In [ ]:
fname = f"degs_{gdc.psi_id}.tsv"
pdwritecsv(df_degs, fname, gdc.root_psi)

### Development and Tests

In [ ]:
def get_file_expression_tumor_and_normal(
    psi_id: str, force: bool = False, verbose: bool = False
) -> Tuple[dict, dict]:
    _, df_tumor_samples, _, _ = gdc.get_filtered_tables(
        psi_id=psi_id, sample_type_term="Primary Tumor", verbose=verbose
    )
    _, df_normal_samples, _, _ = gdc.get_filtered_tables(
        psi_id=psi_id, sample_type_term="Solid Tissue Normal", verbose=verbose
    )

    gdc.file_type_list = np.unique(df_tumor_samples.data_type)

    cols = ["gene_id", "symbol", "gene_type", "counts"]

    dff_normal = df_normal_samples[df_normal_samples.data_type == "Gene Expression Quantification"]
    dff_normal.reset_index(drop=True, inplace=True)

    case_id_list = np.unique(dff_normal.case_id)

    dff_tumor = df_tumor_samples[df_tumor_samples.case_id.isin(case_id_list)].copy()
    dff_tumor = dff_tumor[dff_tumor.data_type == "Gene Expression Quantification"]
    dff_tumor.reset_index(drop=True, inplace=True)

    print(f"There are {len(dff_tumor)} tumor and {len(dff_normal)} normal Gene Expression tables")

    print("Dowloading normal files: ", end="")
    dic_normal = {}
    for i, row in dff_normal.iterrows():
        if int(i) % 10 == 0:
            print(i, end="")
        else:
            print(".", end="")
        case_id = row.case_id
        file_id = row.file_id

        dfexp = gdc.get_table_given_fileID(
            case_id=case_id,
            file_id=file_id,
            sample_type="normal",
            file_type="Gene Expression Quantification",
            verbose=verbose,
            force=force,
        )
        dfexp = dfexp[cols]
        dic_normal[f"normal_{i}"] = dfexp

    print("")

    print("Dowloading tumor files: ", end="")
    dic_tumor = {}
    for i, row in dff_tumor.iterrows():
        if int(i) % 10 == 0:
            print(i, end="")
        else:
            print(".", end="")
        case_id = row.case_id
        file_id = row.file_id

        dfexp = gdc.get_table_given_fileID(
            case_id=case_id,
            file_id=file_id,
            sample_type="tumor",
            file_type="Gene Expression Quantification",
            verbose=verbose,
            force=force,
        )
        dfexp = dfexp[cols]
        dic_tumor[f"tumor_{case_id}"] = dfexp

    print("")

    return dic_tumor, dic_normal

In [ ]:
cols = ["gene_id", "symbol", "gene_type", "counts"]
common_cols = ["gene_id", "symbol", "gene_type"]


dfa_normal = pd.DataFrame()
imax = 13

print(">>> dic_normal", len(dic_normal))
i = 0
for _, dfa in dic_normal.items():
    if dfa is None or dfa.empty:
        continue

    i += 1
    print(i, end=" ")

    dfa = dfa[cols]
    dfa = dfa.rename(columns={"counts": f"normal_{i}"})

    if dfa_normal.empty:
        dfa_normal = dfa
    else:
        if i <= imax:
            dfa_normal = dfa_normal.merge(dfa, on=common_cols, how="outer")
        else:
            print(">>> dfa", len(dfa), ",".join(dfa.symbol[:30]))

print(dfa_normal.shape)
dfa_normal

In [ ]:
dfa_tumor = pd.DataFrame()
imax = 12


print(">>> dic_tumor", len(dic_tumor))
i = 0
for _, dfa in dic_tumor.items():
    if dfa is None or dfa.empty:
        continue

    i += 1
    print(i, end=" ")

    dfa = dfa[cols]
    dfa = dfa.rename(columns={"counts": f"tumor_{i}"})

    if dfa_tumor.empty:
        dfa_tumor = dfa
    else:
        if i <= imax:
            dfa_tumor = dfa_tumor.merge(dfa, on=common_cols, how="outer")
        else:
            print(">>> dfa", len(dfa), ",".join(dfa.symbol[:30]))

dfa_tumor